**Archived exploration.** This notebook records an early project stage, not the current inclusion policy. It requires local research inputs; API collection cells are opt-in when you run them. Saved outputs were removed for publication.


# Step 3 ? Explore, clean and classify candidate advertisements
These are keyword-retrieved Platsbanken candidates, not complete Swedish job-market demand. Rules are provisional until human review. This notebook reads source data and existing processed outputs; it does not overwrite them.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
from src.clean_jobs import latest_collection, load_jobs, normalize_text, publication_day
from src.classify_jobs import CATEGORIES, evaluate_review
collection = latest_collection(ROOT / "data" / "collections")
jobs = load_jobs(collection / "jobs.csv")
print("Collection:", collection.name)
print("Shape:", jobs.shape)
print("Actual columns:", jobs.columns.tolist())

## Missing values and identity checks
CSV fields are loaded as strings without treating literal text such as NA as null. Blank or whitespace-only fields count as missing. Repeated nonempty IDs are duplicates; missing IDs require review. Full original descriptions remain available.

In [ ]:
missing = jobs.apply(lambda column: column.map(normalize_text).eq("").sum())
display(missing.rename("missing_rows").to_frame())
print("Duplicate nonempty IDs:", (jobs.advertisement_id.ne("") & jobs.advertisement_id.duplicated()).sum())
print("Missing descriptions:", jobs.description_text.map(normalize_text).eq("").sum())

## Most common job titles
The 50 most frequent original titles reveal both likely relevant roles and noise introduced by broad searches. Repeated titles do not imply duplicate advertisements.

In [ ]:
display(jobs.job_title.value_counts().head(50).rename_axis("job_title").to_frame("advertisements"))

## Publication dates and locations
Dates use Swedish calendar days, including conversion of timezone-aware timestamps. Monthly and municipality/region tables describe the retrieved candidates only. Missing locations are shown explicitly.

In [ ]:
days = jobs.publication_date.map(publication_day)
print("Invalid/missing dates:", days.isna().sum())
print("Observed range:", days.dropna().min(), days.dropna().max())
monthly = days.dropna().map(lambda d: d.strftime("%Y-%m")).value_counts().sort_index()
display(monthly.rename_axis("publication_month").to_frame("advertisements"))
for column in ["municipality", "region"]:
    display(jobs[column].replace("", "(missing)").value_counts().head(30).rename_axis(column).to_frame("advertisements"))

## Broad AI keyword examples
Use recorded search-term provenance, rather than assuming every AI mention defines an AI role. Examples below include titles outside clear engineering roles and show description excerpts. Search terms overlap and their counts are not additive.

In [ ]:
broad = {"ai engineer", "ai developer", "ai-utvecklare", "ai-ingenj?r", "artificiell intelligens", "maskininl?rning"}
mask = jobs.search_terms.map(lambda value: bool(set(json.loads(value or "[]")) & broad))
print("Candidates retrieved by these terms:", mask.sum())
examples = jobs.loc[mask].sample(n=min(12, int(mask.sum())), random_state=42).copy()
examples["description_excerpt"] = examples.description_text.str.slice(0, 500)
with pd.option_context("display.max_colwidth", 500):
    display(examples[["advertisement_id", "job_title", "search_terms", "description_excerpt"]])

## Cleaning and classification output
Run `python -m src.classify_jobs` from the project root first. Original columns are preserved; normalized text, date status, primary category, method, evidence and review status are appended. Exact nonempty duplicate IDs keep their first occurrence. Missing IDs stay separate. Multi-role and generic titles are flagged; description mentions alone never create an AI Engineer classification. Invalid dates stay in output but are excluded from category counts.

Set `processed_dir` explicitly to revisit an earlier run. Each pipeline invocation creates a fresh folder so completed review files are preserved.

In [ ]:
processed_dir = max((ROOT / "data" / "processed").glob("*/classification_summary.json")).parent
summary = json.loads((processed_dir / "classification_summary.json").read_text(encoding="utf-8"))
print("Processed run:", processed_dir)
display(pd.Series(summary["count_per_job_category"], name="advertisements").to_frame())
print(summary["coverage_note"])
classified = pd.read_csv(processed_dir / "jobs_classified.csv", dtype=str, keep_default_na=False)
display(classified[["job_title", "primary_job_category", "classification_method", "classification_evidence", "review_status"]].sample(15, random_state=42))

## Manual review and validation
Open this run?s `manual_review.csv` in Excel or another CSV editor. Read each original title and full description, then fill `correct_category` using exactly one category below; use `Uncertain / Needs Review` if the evidence cannot resolve the role. Record reasoning in `review_notes`. Do not edit IDs, predictions or sampling metadata. Save as UTF-8 CSV. `pending_validation` is a prediction, not a human approval.

The seeded balanced sample randomly orders rows within predicted-category strata, then draws round-robin. Small strata are fully included. It includes uncertain cases and records population/sample stratum sizes. This deliberately overrepresents rare categories. Accuracy below applies only to supplied labels in this sample, not the full corpus; partial or selective review can add bias. No accuracy is calculated before human labels exist. Rows with invalid dates, if present, are reviewable but excluded from analysis counts.

In [ ]:
print("Allowed labels:")
print("\n".join(CATEGORIES))
review_path = processed_dir / "manual_review.csv"
review = pd.read_csv(review_path, dtype=str, keep_default_na=False)
display(review.primary_job_category.value_counts().to_frame("review_rows"))
metrics, confusion = evaluate_review(review)
if metrics["reviewed"] == 0:
    print("No human labels supplied. Accuracy is not yet available.")
else:
    print(metrics)
    display(confusion)
    display(review.loc[review.correct_category.ne("") & review.correct_category.ne(review.primary_job_category),
                       ["advertisement_id", "job_title", "primary_job_category", "correct_category", "review_notes"]])